# Fine-tune DARE3D on your own data

**Fine-tuning** (a form of *transfer learning*) takes a model already trained on one dataset and continues training it — gently — on *your* data. Instead of learning from scratch (which needs a lot of labelled data and time), you start from a model that already "knows" what a dividing cell looks like and only nudge it to fit your images.

By the end of this notebook you'll have a fine-tuned model saved in the exact format DARE3D uses for inference, ready to detect divisions in your movies.

**A few terms up front:**
- **Checkpoint** — a saved model: its learned weights plus the recipe (`.hydra/config.yaml`) describing how it was built. You fine-tune *from* a checkpoint.
- **Backbone / head** — the *backbone* is the bulk of the network that extracts features; the *head* is the small output part. In fine-tuning we usually **freeze** the backbone (keep its weights fixed) and retrain only the head, so the model keeps what it already learned.
- **Stage** — DARE3D works in two networks: **segmentation** (finds the *center* of each division) and **regression** (estimates the division *axis*). You fine-tune one stage at a time.

**When should you fine-tune?** When the released models don't quite work on your data (different microscope, marker, species, resolution) but you have a modest amount of your own labelled movies — too few to train from scratch.

> You can also fine-tune from the **napari widget** (*DARE3D retraining & fine-tuning*) or the **command line** — they run exactly the same training. This notebook is the recommended, most transparent way to learn what's happening.

**Roadmap:** setup → understand which settings you may change → pick your data and base model → set hyperparameters → **run the fine-tune** → **use your new model**. An optional appendix at the end opens the hood and verifies each mechanic.

## Before you touch any knob: three kinds of setting

A checkpoint is not just weights — it is weights **of a specific shape**, trained on data of a specific **geometry**, with a specific **optimization recipe**. When you fine-tune, some settings are *dictated by the checkpoint you start from* and some are *yours to choose*. Getting this distinction wrong is the classic way fine-tuning fails: the model either refuses to load, or loads and quietly transfers badly.

DARE3D sorts every fine-tuning setting into exactly **three buckets**. Keep this mental model as you go — each section below is labelled with the bucket it configures.

### 1. Imposed — fixed by the checkpoint (you don't choose these)
The pretrained weights have fixed shapes, so the network that receives them must be built identically: its type, width and depth, the number of segmentation output classes, the normalization type — and the input **geometry** *only if* the network has a size-dependent layer (e.g. a dense head that flattens a fixed-size volume). Change any of these and loading fails with a shape-mismatch error.

You don't set these by hand. **This notebook reads them straight out of your base checkpoint** and shows you the exact layer that fixes each one, so the model is correct by construction.

### 2. Runtime-free — yours to choose (optimization + fully-convolutional geometry)
These don't change any weight shape, so you're free to set them: the learning rate and its schedule, weight decay, batch size, augmentation, early-stopping patience, the random seed — **plus** the input geometry (`seg_crop_size`, `cell_radius`) *when the network is fully-convolutional* (has no size-dependent layer).

"Free" doesn't mean "consequence-free": a crop size far from what the base was trained on still *loads*, but transfers worse; the *from-scratch* learning rate is far too aggressive for weights that start already-trained. The notebook prints the **base's own training value next to each of these** so you can spot any divergence before you run.

### 3. Fine-tune-only — the transfer-learning recipe
Settings that only mean something when you start from a pretrained model: **what to freeze** (`freeze_preset`), **how deep to thaw** (`unfreeze_last_stages`), and **how to treat the frozen network's BatchNorm** (`bn_mode`). Explained in section 4 where you set them.

## Setup

Imports, the repository root, and the cuDNN gate (DARE3D's 3-D training needs a healthy cuDNN — this env is `dare3d-v2.0`, torch 2.5 / cuDNN 9.1). `verdict()` is a small helper the optional appendix uses to print PASS/FAIL check results; you can ignore it for now.

In [ ]:
# --- Setup -----------------------------------------------------------------
# cuDNN gate: DARE3D 3D-convolution training segfaults on old cuDNN (8.x); set DARE3D_CUDNN=1 to keep
# cuDNN ON on a healthy stack (this env = dare3d-v2.0, torch 2.5 / cuDNN 9.1). See dare3d/train.py.
import os, sys, json, tempfile, warnings
from pathlib import Path
os.environ.setdefault("DARE3D_CUDNN", "1")

# Robust repo root: the folder holding dare3d/ + configs/ (walk up from cwd).
def _find_repo(start: Path) -> Path:
    for c in (start.resolve(), *start.resolve().parents):
        if (c / "dare3d").is_dir() and (c / "configs").is_dir():
            return c
    return start.resolve()
REPO = _find_repo(Path.cwd())
sys.path.insert(0, str(REPO))
os.environ.setdefault("PROJECT_ROOT", str(REPO))
CONFIGS = REPO / "configs"

import torch
from omegaconf import OmegaConf
try:
    OmegaConf.register_new_resolver("eval", eval, replace=True)   # dare3d configs use ${eval:...}
except Exception:
    pass
from napari_dare3d import _train                       # the widget's dispatch (napari-free)
from dare3d.models.finetune import load_net_state_dict, apply_freeze, FineTuneError

# Imported here so the fine-tune run and the appendix diagnostics can all use them.
import lightning as L
from lightning.pytorch import Callback
from lightning.pytorch.plugins.environments import SLURMEnvironment
SLURMEnvironment.detect = staticmethod(lambda: False)   # never treat the notebook as a SLURM job
from hydra import initialize_config_dir, compose
from hydra.utils import instantiate

RESULTS = {}                                            # check-name -> "PASS"/"FAIL"/"N/A: reason"
def verdict(name, ok, extra=""):
    v = "PASS" if ok is True else ("FAIL" if ok is False else str(ok))
    RESULTS[name] = v
    print(f"[{v}] {name}" + (f"  ({extra})" if extra else ""))
    return ok

import lightning
print("repo        :", REPO)
print("torch       :", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| cuDNN:", torch.backends.cudnn.version())
print("lightning   :", lightning.__version__, "| DARE3D_CUDNN:", os.environ.get("DARE3D_CUDNN"))

## 1) Choose your base model, stage, and data

Three choices define your fine-tuning job:

- **`BASE_CKPT`** — the checkpoint you adapt (a model directory's `checkpoints/last.ckpt`). Your starting point.
- **`STAGE`** — `"regression"` (the division-axis network) or `"segmentation"` (the center-detection U-Net). One checkpoint per stage; fine-tune each separately.
- **Your data** — movies laid out as `<dataset>/<movie>/{im,label}/*.tif` (one 4-D `(T,Z,Y,X)` movie per folder, plus matching labels). You choose which movies **train** the model and which are **held out for validation** — kept separate so the validation score reflects data the model never trained on (your honest measure of whether fine-tuning helped).

**To fine-tune on *your* data:** point `DATA`/`DATASET` at your folder and list your movie names in `TRAIN_MOVIES`/`VAL_MOVIES`. Here we default to the published **Gastruloid** regression model as a small, fast example (its network contains BatchNorm, which the appendix's frozen-BN check needs). For segmentation, set `STAGE="segmentation"` and point `BASE_*` at a seg model directory.

In [ ]:
STAGE = "regression"                      # "regression" or "segmentation"
DATA = REPO / "DARE3dv2_Zenodo_040926" / "Gastruloid_241025"
BASE_DIR  = DATA / "weights" / (f"{STAGE}3d_exp10-b")
BASE_CKPT = BASE_DIR / "checkpoints" / "last.ckpt"
DATASET   = DATA / "trainingset"
TRAIN_MOVIES, VAL_MOVIES = ["movie3"], ["movie2"]   # disjoint specimens (val = held-out)

assert BASE_CKPT.is_file(), f"base checkpoint not found: {BASE_CKPT} (run dare3d-download to fetch the Zenodo bundle)"
assert DATASET.is_dir(), f"dataset not found: {DATASET}"

# A private, disposable output dir for this notebook's runs (never touches the repo).
OUT = Path(tempfile.mkdtemp(prefix="dare3d_finetune_nb_"))
SPLIT = _train._resolve_split(str(DATASET), str(OUT), TRAIN_MOVIES, VAL_MOVIES)
print(f"stage      = {STAGE}")
print(f"base_ckpt  = {BASE_CKPT}")
print(f"dataset    = {DATASET}  (train={TRAIN_MOVIES}, val={VAL_MOVIES})")
print(f"split      = {SPLIT}")
print(f"out (temp) = {OUT}")

## 2) Read the architecture from the checkpoint  ·  *imposed*

You can't choose the network's shape — the pretrained weights require an exact match. The next cell opens the base checkpoint's saved config and its weights and **reads off** the architecture (network type, channel widths, depth, output classes, normalization), printing **the specific layer/shape that fixes each value**.

It also settles the geometry question *empirically*: it scans the weights for any 2-D (`Linear`) weight — a size-dependent dense head. If there is none, the network is **fully convolutional**, so the input crop size is a *runtime-free* choice (bucket 2); if there is one, the crop is baked into that layer and is therefore *imposed*.

These land in a read-only `IMPOSED` dict. **Don't edit them** — when you launch the fine-tune, the same values are re-derived from the base and passed to the network, so the load matches by construction.

In [ ]:
cfg_base = OmegaConf.load(BASE_DIR / ".hydra" / "config.yaml")
sd = torch.load(BASE_CKPT, map_location="cpu", weights_only=False)["state_dict"]
net_sd = {k[len("net."):]: v for k, v in sd.items() if k.startswith("net.")}
sel = lambda k: OmegaConf.select(cfg_base, k)

net_class = str(sel("model.net._target_")).split(".")[-1]
linear_layers = [(k, tuple(v.shape)) for k, v in net_sd.items() if v.ndim == 2]   # nn.Linear
conv3d = sum(1 for v in net_sd.values() if v.ndim == 5)

IMPOSED = {"net_class": net_class,
           "start_filters": sel("model.net.start_filters"), "n_stages": sel("model.net.n_stages"),
           "channels": sel("model.net.channels"), "strides": sel("model.net.strides"),
           "out_channels": sel("model.net.out_channels"), "norm": sel("model.net.norm"),
           "base_crop_size": sel("crop_size"), "base_cell_radius": sel("cell_radius")}

print(f"IMPOSED architecture (from base {BASE_DIR.name}):")
print(f"  net_class      = {net_class!r}   ({conv3d} Conv3d weights, {len(linear_layers)} Linear weights)")
for k in ("start_filters", "n_stages", "channels", "strides", "out_channels", "norm"):
    if IMPOSED[k] is not None:
        print(f"  model.net.{k:<13}= {IMPOSED[k]}")

print("\nForcing layers (the shape that fixes each imposed value):")
if linear_layers:
    for name, shp in linear_layers:
        print(f"  SIZE-DEPENDENT HEAD  net.{name:<20} nn.Linear{shp}  -> in_features={shp[1]} is fixed by "
              f"start_filters*2^(n_stages-1) * (crop/2^n_stages)^3")
    print(f"  => geometry is IMPOSED for this net (crop is baked into the dense head). This repo fixes the "
          f"regression crop at {IMPOSED['base_crop_size']} and never exposes it, so there is no free knob.")
    GEOMETRY = "imposed"
else:
    prod = 1
    for s in (IMPOSED["strides"] or []):
        prod *= int(s[0] if isinstance(s, (list, tuple)) else s)
    print(f"  final Conv3d out_channels={IMPOSED['out_channels']} (segmentation head classes) — fixed by the last conv")
    print(f"  U-Net channels/strides/norm — fixed by every Conv3d/BatchNorm3d weight")
    print(f"  NO 2-D (Linear) weight anywhere -> the net is FULLY CONVOLUTIONAL -> the input crop is RUNTIME")
    print(f"  (only a HARD constraint remains: seg_crop_size must be divisible by prod(strides)={prod})")
    GEOMETRY = "runtime"
print(f"\nGEOMETRY verdict for {STAGE}: {GEOMETRY}")

## 3) Set your optimization hyperparameters  ·  *runtime-free*

These are yours to choose — they don't change any weight shape. Sensible fine-tuning values are set below; the cell prints each against the base's own training value so divergence is visible before you run.

**What each one means, and when to change it:**
- **`ft_lr` — learning rate (default `1e-4`).** How big a step the optimizer takes. From-scratch training uses large rates (≈1e-2–1e-1) that would *erase* the pretrained features in a few steps — keep fine-tuning at ≤1e-3. This is the knob you're most likely to tune.
- **`backbone_lr_mult` + `discriminative`.** If you thaw part of the backbone, train it *slower* than the head (backbone rate = `ft_lr × backbone_lr_mult`). Turn `discriminative` off to use one rate everywhere.
- **`lr_schedule` / `warmup_epochs`.** `warmup_cosine` ramps the rate up over `warmup_epochs`, then eases it down a cosine curve — a gentle start that avoids jolting the pretrained weights.
- **`weight_decay`.** Mild regularization (discourages large weights); `1e-4` is standard.
- **`batch_size`.** How many 3-D patches per step; lower it if you hit GPU out-of-memory. Interacts with `bn_mode` (next section): `adapt` + a tiny batch = noisy BatchNorm statistics.
- **`patience` / `seed`.** Early-stop after `patience` epochs with no validation improvement; `seed` makes the run reproducible.
- **`augment` / `augment_strength`.** Data augmentation and the per-sample probability of applying it. With a fully-frozen backbone, heavy augmentation can hurt — only the small head can adapt to it.
- **geometry (`seg_crop_size`, `cell_radius`) — segmentation only.** Fully-convolutional, so any value *loads*, but staying near the base's training value transfers best. **Inert for regression** (its crop is imposed-fixed, section 2).

In [ ]:
# --- runtime-free: EDIT THESE FREELY ---
ft_lr            = "1e-4"
backbone_lr_mult = "0.1"
discriminative   = True
weight_decay     = "1e-4"
lr_schedule      = "warmup_cosine"     # "warmup_cosine" (warmup->cosine) or "cosine"
warmup_epochs    = 2
grad_clip        = "1.0"               # gradient max-norm; "0" = off
patience         = 10                  # early-stop epochs w/o val improvement
seed             = 12345
augment          = True
augment_strength = "0.5"               # per-sample augmentation probability (0-1); 0 = none
batch_size       = 4
# geometry (segmentation only; inert for regression):
seg_crop_size    = 128
cell_radius      = 8

print("runtime-free knobs vs the base's training values:")
base_crop = IMPOSED["base_crop_size"]; base_radius = IMPOSED["base_cell_radius"]
if STAGE == "segmentation":
    print(f"  seg_crop_size = {seg_crop_size:<5} | base trained at crop = {base_crop}"
          + ("   <-- DIVERGES: receptive-field/transfer risk" if base_crop is not None and seg_crop_size != base_crop else "   (matches base)"))
    print(f"  cell_radius   = {cell_radius:<5} | base trained at radius = {base_radius}"
          + ("   <-- DIVERGES: target-scale shift" if base_radius is not None and cell_radius != base_radius else "   (matches base)"))
else:
    print(f"  (regression) seg_crop_size / cell_radius are inert; the regression crop is imposed at {base_crop}.")
print(f"  ft_lr = {ft_lr}  (fine-tune scale; <=1e-3 recommended, NOT the scratch LR)")
print(f"  batch_size = {batch_size}  x  bn_mode below  (adapt + tiny batch = noisy BN stats)")

## 4) Choose the freeze recipe  ·  *fine-tune-only*

This is the heart of fine-tuning: **what stays fixed and what learns.** A network has a **backbone** (the feature extractor — an *encoder* that compresses the image and, for the U-Net, a *decoder* that expands it) and a small **head** that produces the final output.

- **`freeze_preset`**
  - `encoder` (default) — freeze the backbone, train only the decoder/head. The safest, most common choice: keep everything the base learned, adapt only the output.
  - `encoder_partial` — also thaw the last `unfreeze_last_stages` stages of the backbone, for a bit more adaptation.
  - `none` — train everything (full fine-tune); use only with plenty of data.
- **`unfreeze_last_stages`** — with `encoder_partial`, how many of the deepest backbone stages to unfreeze.
- **`bn_mode` — how to treat the frozen backbone's BatchNorm.** *(BatchNorm layers normalize activations using running statistics gathered during training.)*
  - `frozen` (default) — those statistics stay fixed: a *true* freeze. Recommended.
  - `adapt` — let them re-estimate on your data (the weights still stay frozen) — for a larger, visibly different fine-tuning set.

> **Why `bn_mode` is subtle (and handled correctly here).** A "frozen" BatchNorm must stay frozen *even during the short validation passes Lightning runs mid-epoch*. DARE3D enforces this at the right level so the statistics never quietly drift — the appendix demonstrates it across a real mid-epoch validation boundary.

In [ ]:
# --- fine-tune-only: EDIT THESE FREELY ---
freeze_preset        = "encoder"        # encoder | encoder_partial | none
unfreeze_last_stages = 0                # >0 with encoder_partial
bn_mode              = "frozen"         # frozen (true freeze) | adapt (re-estimate running stats)

print(f"freeze_preset        = {freeze_preset}")
print(f"unfreeze_last_stages = {unfreeze_last_stages}")
print(f"bn_mode              = {bn_mode}")

## 5) Build the training command

We collect your choices into one `ft` dictionary and hand it to a small helper, `finetune_command(...)`, which assembles the exact `dare3d/train.py` invocation (a list of Hydra overrides). The *imposed* `model/net=…` settings are added automatically from the base.

For reference, here is how each notebook variable maps to what it controls:

| notebook variable | what it sets | bucket |
|---|---|---|
| `BASE_CKPT`, `STAGE` | base checkpoint & which network | selects the job |
| *(derived)* net class, channels, strides, out_channels, norm, … | architecture | **imposed** |
| `freeze_preset`, `unfreeze_last_stages`, `bn_mode` | the freeze recipe | **fine-tune-only** |
| `ft_lr`, `discriminative`/`backbone_lr_mult`, `weight_decay` | optimizer | runtime-free |
| `lr_schedule`/`warmup_epochs`, `grad_clip`, `patience`, `seed` | schedule & stopping | runtime-free |
| `augment`/`augment_strength`, `batch_size` | data | runtime-free |
| `seg_crop_size`/`cell_radius` (seg only) | geometry | runtime-free (fully-conv) |

The command printed below uses **`epochs=50`** — that is a real fine-tuning run. (Further down we execute a *tiny capped* version so the notebook finishes in seconds; to fine-tune for real, run the printed command in a terminal, or raise the caps.)

In [ ]:
ft = dict(freeze_preset=freeze_preset, unfreeze_last_stages=int(unfreeze_last_stages), bn_mode=bn_mode,
          ft_lr=ft_lr, discriminative=bool(discriminative), backbone_lr_mult=backbone_lr_mult,
          weight_decay=weight_decay, lr_schedule=lr_schedule, warmup_epochs=int(warmup_epochs),
          grad_clip=grad_clip, augment=bool(augment), augment_strength=augment_strength,
          patience=int(patience), seed=int(seed), cell_radius=int(cell_radius),
          seg_crop_size=int(seg_crop_size))

# pre-flight (same as the widget): errors block, warnings would prompt. Shown for transparency.
pf = _train.finetune_preflight([STAGE], {STAGE: str(BASE_CKPT)}, ft)
print("pre-flight errors  :", pf["errors"] or "(none)")
print("pre-flight warnings:", pf["warnings"] or "(none)")

cmd = _train.finetune_command(STAGE, str(SPLIT), str(OUT), "demo", "demo", str(BASE_CKPT), ft,
                              epochs=50, batch_size=int(batch_size))
print("\nResolved Hydra overrides (identical to the widget's dispatch):")
for tok in cmd[2:]:
    print("  ", tok)

## 6) Run the fine-tune

This launches `dare3d/train.py` as a subprocess — the real training entry point — streaming its log here. It writes a **model directory** containing `checkpoints/last.ckpt` (your fine-tuned model) and a `finetune_config.json` recording exactly how it was produced.

To keep the notebook fast we cap it to a couple of batches for one epoch, so the numbers below are *not* a converged model — they show the pipeline runs and the result is usable. **For a real run, use the `epochs=50` command printed above** (in a terminal, or by raising the caps). After training we reload the saved checkpoint through DARE3D's **actual inference loader** and run a forward pass, confirming your fine-tuned model drops straight into prediction.

In [ ]:
cmd_sub = _train.finetune_command(STAGE, str(SPLIT), str(OUT), "nbsub", "nbsub", str(BASE_CKPT),
                                  ft, epochs=1, batch_size=2) + ["+trainer.limit_train_batches=3",
                                                                 "+trainer.limit_val_batches=2"]
print("subprocess:", " ".join(cmd_sub[2:5]), "...\n")
try:
    n = 0
    for line in _train._stream(cmd_sub, None):
        n += 1
        if n <= 2 or n % 60 == 0: print("  |", line[:105])
    stream_ok = True
except Exception as e:
    stream_ok = False; print("  subprocess error:", repr(e))
model_dir = _train.model_dirs(str(OUT), "nbsub", "nbsub")[STAGE]
ckpt = model_dir / "checkpoints" / "last.ckpt"
sidecar = model_dir / "checkpoints" / "finetune_config.json"
verdict("B1 subprocess produced ckpt + sidecar", stream_ok and ckpt.is_file() and sidecar.is_file(),
        f"stream_ok={stream_ok}, ckpt={ckpt.is_file()}, sidecar={sidecar.is_file()}")

# round-trip through the REAL inference loader (predict.py::load_data pattern)
try:
    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    cfg_ft = OmegaConf.load(model_dir / ".hydra" / "config.yaml")
    m2 = instantiate(cfg_ft.model)
    m2.load_state_dict(torch.load(ckpt, map_location="cpu", weights_only=False)["state_dict"])
    m2.net = m2.net.to(dev); m2.net.eval()
    crop = int(sel("crop_size")) if STAGE == "regression" else 64
    with torch.no_grad():
        out = m2.net(torch.randn(1, 3, crop, crop, crop, device=dev))
    ok_fwd = isinstance(out, dict) and len(out) > 0
    verdict("B2 round-trip: fine-tuned ckpt reloads via the inference loader + forward runs", ok_fwd,
            f"output keys={list(out.keys()) if isinstance(out, dict) else type(out)}")
except Exception as e:
    verdict("B2 round-trip: fine-tuned ckpt reloads via the inference loader + forward runs", False, f"error: {e!r}")

# provenance sidecar reflects the GUI/notebook values
try:
    info = json.loads(sidecar.read_text()); hp = info["hyperparameters"]
    sc_ok = (hp["bn_mode"] == ft["bn_mode"] and hp["freeze_preset"] == ft["freeze_preset"]
             and abs(float(hp["ft_lr"]) - float(ft["ft_lr"])) < 1e-9
             and info["base_model"]["sha256"] and info["environment"]["torch"])
    verdict("B3 sidecar records the run's provenance + hyperparameters", bool(sc_ok),
            f"bn_mode={hp['bn_mode']}, freeze_preset={hp['freeze_preset']}, ft_lr={hp['ft_lr']}, "
            f"base_sha256={str(info['base_model']['sha256'])[:12]}, torch={info['environment']['torch']}")
except Exception as e:
    verdict("B3 sidecar records the run's provenance + hyperparameters", False, f"error: {e!r}")

## 7) Use your fine-tuned model

Your fine-tuned model lives in its **model directory** (the folder with `.hydra/config.yaml` + `checkpoints/`). Point any DARE3D inference path at it, exactly as you would a released model:

- **napari** — *DARE3D inference* widget; set the Segmentation/Regression model-dir field.
- **notebook** — `Run_dare3d_Prediction.ipynb`; set the model dir.
- **CLI** — `python dare3d/predict.py +segmentation.model_dir=… +regression.model_dir=… +inference_dir=…`.

Because the fine-tuned checkpoint is byte-compatible with the inference loader (just confirmed by the reload above), there is nothing else to convert. **That is the whole workflow** — everything below is an optional look under the hood.

---
## Appendix — under the hood (optional)

You already know how to fine-tune; you can stop here. These cells **verify the mechanics**, for the curious or the cautious: that the base weights really loaded, the freeze really froze, the optimizer really uses two learning rates, and the frozen BatchNorm really holds across a mid-epoch validation. Each prints real numbers and a **PASS / FAIL / N/A** verdict (collected by the `verdict()` helper from Setup).

### A. Watch the mechanics live — load the base

One tiny real `trainer.fit`, run *inside this process* so we can read live internals with a callback. For this diagnostic only we thaw one stage (`encoder_partial`) so there are two optimizer groups to inspect, and we validate **mid-epoch** (`val_check_interval=0.5`) — the exact moment a broken frozen-BN would slip.

This first cell composes the config, instantiates the model, calls `model.load_base()`, and runs check **(A)**: the base weights landed in `self.net`, are equal to the base, and differ from a fresh random init.

In [ ]:
# same dispatch overrides, but encoder_partial (to get 2 optimizer groups) + tiny caps
ft_fit = dict(ft, freeze_preset="encoder_partial", unfreeze_last_stages=1, warmup_epochs=1)
cmd_fit = _train.finetune_command(STAGE, str(SPLIT), str(OUT), "nbfit", "nbfit", str(BASE_CKPT),
                                  ft_fit, epochs=2, batch_size=2)
(OUT / "inproc").mkdir(parents=True, exist_ok=True)
# paths.output_dir is ${hydra:runtime.output_dir} in the config -> only resolves under hydra.main;
# under compose() we pin it (and work_dir) to a concrete temp path so Trainer can instantiate.
overrides = cmd_fit[2:] + ["+trainer.limit_train_batches=4", "+trainer.limit_val_batches=2",
                           "+trainer.val_check_interval=0.5", "data.num_workers=0",
                           "trainer.deterministic=warn", "+trainer.enable_progress_bar=false",
                           f"paths.output_dir={(OUT / 'inproc').as_posix()}",
                           f"paths.work_dir={OUT.as_posix()}"]
with initialize_config_dir(config_dir=str(CONFIGS), version_base="1.3"):
    cfg = compose(config_name="train", overrides=overrides)

model = instantiate(cfg.model)
fresh = {k: v.detach().cpu().clone() for k, v in model.net.state_dict().items()}   # random init
model.load_base()                                                                 # engine: base -> self.net
loaded = {k: v.detach().cpu().clone() for k, v in model.net.state_dict().items()}
base_net = {k[len("net."):]: v for k, v in
            torch.load(BASE_CKPT, map_location="cpu", weights_only=False)["state_dict"].items()
            if k.startswith("net.")}

# (A) weights loaded, key-complete, == base, != fresh
missing = [k for k in base_net if k not in loaded]
eq_base = all(torch.equal(loaded[k], base_net[k]) for k in base_net if k in loaded)
diff_fresh = sum(1 for k in fresh if not torch.equal(fresh[k], loaded[k]))
verdict("A weights loaded into self.net (key-complete, ==base, !=fresh)",
        (not missing) and eq_base and diff_fresh > 0,
        f"missing={len(missing)}, equal_to_base={eq_base}, changed_from_fresh={diff_fresh}/{len(fresh)}")


### …then fit and check B–E

Now the actual `trainer.fit`, with a `Probe` callback recording internals at every step. Checks: **(B)** frozen params unchanged / trainable params moved; **(C)** the optimizer has the two discriminative param-groups; **(D)** data shapes `(B,C,Z,Y,X)`, train/val disjoint, inputs normalized; **(E)** the frozen `BatchNorm3d` stayed in `.eval()` and its running stats did not move across the mid-epoch validation boundary (N/A if the net has no BatchNorm3d).

In [ ]:
class Probe(Callback):
    def __init__(self): self.rec = []; self.bn = None; self.val_starts = 0
    def on_train_start(self, tr, pl):
        bns = [m for m in pl.net.modules() if isinstance(m, torch.nn.BatchNorm3d)
               and not any(p.requires_grad for p in m.parameters())]
        self.bn = bns[0] if bns else None
    def on_train_batch_start(self, tr, pl, batch, bi):
        if self.bn is not None:
            self.rec.append((pl.training, self.bn.training, self.bn.running_mean.detach().float().cpu().clone()))
    def on_validation_start(self, tr, pl): self.val_starts += 1

probe = Probe()
dm = instantiate(cfg.data)
trainer = instantiate(cfg.trainer, logger=False, enable_checkpointing=False, callbacks=[probe])
L.seed_everything(int(ft_fit["seed"]), workers=True)
trainer.fit(model, dm)

# (B) freeze deltas: frozen == base (unchanged), trainable != base (moved)
frozen_ok, trainable_moved = True, False
for n, p in model.net.named_parameters():
    if n not in base_net: continue
    same = torch.equal(p.detach().cpu(), base_net[n])
    if p.requires_grad:
        trainable_moved = trainable_moved or (not same)
    else:
        frozen_ok = frozen_ok and same
n_tr = sum(int(p.requires_grad) for p in model.net.parameters())
verdict("B freeze real (frozen unchanged vs base, trainable changed)", frozen_ok and trainable_moved,
        f"trainable_tensors={n_tr}, frozen_unchanged={frozen_ok}, trainable_moved={trainable_moved}")

# (C) optimizer param-groups + discriminative LR
try:
    opt = trainer.optimizers[0]
    base_lrs = [pg.get("initial_lr", pg["lr"]) for pg in opt.param_groups]
    ratio = (min(base_lrs) / max(base_lrs)) if max(base_lrs) else 0.0
    verdict("C optimizer has discriminative param-groups", len(opt.param_groups) == 2
            and abs(ratio - float(ft_fit["backbone_lr_mult"])) < 1e-6,
            f"groups={len(opt.param_groups)}, base_lrs={base_lrs}, ratio={ratio:.3g} (mult={ft_fit['backbone_lr_mult']})")
except Exception as e:
    verdict("C optimizer has discriminative param-groups", False, f"error: {e!r}")

# (D) data shapes / disjoint / normalization
try:
    xb = next(iter(dm.train_dataloader()))[0]["input"]
    tr_im = set(os.listdir(SPLIT / "train" / "im")); va_im = set(os.listdir(SPLIT / "val" / "im"))
    rng = (float(xb.min()), float(xb.max()))
    verdict("D data (B,C,Z,Y,X), train/val disjoint, normalized",
            xb.ndim == 5 and xb.shape[1] == 3 and tr_im.isdisjoint(va_im) and -0.01 <= rng[0] and rng[1] <= 1.01,
            f"shape={tuple(xb.shape)}, disjoint={tr_im.isdisjoint(va_im)}, range=({rng[0]:.3f},{rng[1]:.3f})")
except Exception as e:
    verdict("D data (B,C,Z,Y,X), train/val disjoint, normalized", False, f"error: {e!r}")

# (E) frozen BatchNorm3d across a genuine mid-epoch validation boundary
try:
    if probe.bn is None:
        verdict("E frozen-BN survives mid-epoch validation", "N/A: net has no frozen BatchNorm3d")
    else:
        eval_ok = all((not bn_tr) for (mod_tr, bn_tr, _) in probe.rec if mod_tr)
        rm0 = probe.rec[0][2]
        stats_frozen = all(torch.equal(rm0, r[2]) for r in probe.rec)
        crossed = probe.val_starts >= 2
        verdict("E frozen-BN eval + stats fixed across mid-epoch val", eval_ok and stats_frozen and crossed,
                f"records={len(probe.rec)}, always_eval_in_train={eval_ok}, stats_unchanged={stats_frozen}, "
                f"val_boundaries={probe.val_starts}")
except Exception as e:
    verdict("E frozen-BN survives mid-epoch validation", False, f"error: {e!r}")

### B. The imposed values really bind

Proof that the architecture the notebook *derived* (section 2) is the one the weights require: we rebuild the network with a **wrong** value (halved `start_filters` / channels) and show the load is rejected, while the derived value loads clean and its head shape matches.

In [ ]:
if STAGE == "regression":
    from dare3d.models.components.simple_regression_net import RegressionNet
    good = RegressionNet(input_channels=[-1, 0, 1], im_size=int(IMPOSED["base_crop_size"]),
                         n_stages=int(IMPOSED["n_stages"]), start_filters=int(IMPOSED["start_filters"]))
    load_net_state_dict(good, str(BASE_CKPT), stage=STAGE)                    # inherited -> clean
    head_in = good.head1_len[0].in_features
    wrong = RegressionNet(input_channels=[-1, 0, 1], im_size=int(IMPOSED["base_crop_size"]),
                          n_stages=int(IMPOSED["n_stages"]), start_filters=int(IMPOSED["start_filters"]) // 2)
    try:
        load_net_state_dict(wrong, str(BASE_CKPT), stage=STAGE); raised = False
    except FineTuneError:
        raised = True
    verdict("C imposed constraints bind (derived loads clean; wrong start_filters rejected)",
            raised and head_in == linear_layers[0][1][1],
            f"derived head in_features={head_in} (==Linear shape {linear_layers[0][1]}); "
            f"wrong start_filters rejected={raised}")
else:
    # segmentation: the derived channels/strides load; a wrong channel width is rejected
    from dare3d.models.components.multiscale_unet import MultiScaleUNet
    ch = list(IMPOSED["channels"]); st = list(IMPOSED["strides"])
    good = MultiScaleUNet(spatial_dims=3, in_channels=3, out_channels=int(IMPOSED["out_channels"]),
                          channels=ch, strides=st, norm=IMPOSED["norm"], num_res_units=3, bias=False,
                          dropout=0.0, output_names=[], downsample_factors=[1])
    load_net_state_dict(good, str(BASE_CKPT), stage=STAGE)
    wrong = MultiScaleUNet(spatial_dims=3, in_channels=3, out_channels=int(IMPOSED["out_channels"]),
                           channels=[c // 2 for c in ch], strides=st, norm=IMPOSED["norm"],
                           num_res_units=3, bias=False, dropout=0.0, output_names=[], downsample_factors=[1])
    try:
        load_net_state_dict(wrong, str(BASE_CKPT), stage=STAGE); raised = False
    except FineTuneError:
        raised = True
    verdict("C imposed constraints bind (derived channels load; halved channels rejected)", raised,
            f"wrong channels rejected={raised}")

### Verification summary

The consolidated PASS / FAIL / N/A table for every check above.

In [ ]:
print("=== DARE3D fine-tune notebook — verification summary ===")
for k, v in RESULTS.items():
    print(f"  {v:<6} {k}")
n_fail = sum(1 for v in RESULTS.values() if v == "FAIL")
print(f"\n{sum(1 for v in RESULTS.values() if v=='PASS')} PASS / "
      f"{n_fail} FAIL / {sum(1 for v in RESULTS.values() if v.startswith('N/A'))} N/A")
print("ALL GOOD" if n_fail == 0 else "SOME CHECKS FAILED — see above")